# Day 9 · 数据合成

**配套讲义**: `days/day-09.md` ｜ **本地可跑，需要 `.env` 里的 API key**
⚠️ 今天花真金白银。流程：20 条试跑 → 抽检 ≥70% → 才放量 2k。

## 0. API key 就位检查

In [ ]:
import sys; sys.path.insert(0, ".."); os_probe = True
from pathlib import Path
env = Path("../.env")
print("`.env` 存在:", env.exists())
if env.exists():
    keys = [l.split("=")[0] for l in env.read_text().splitlines()
            if "=" in l and not l.strip().startswith("#")]
    print("已定义的变量:", keys)
    print("SYNTH_API_KEY 有值:", any("SYNTH_API_KEY=" in l and l.split("=",1)[1].strip()
                                    for l in env.read_text().splitlines()))

## 1. 试跑 20 条（几分钱）

In [ ]:
# 终端里跑更稳（notebook 里长任务不好中断）:
#   python -m src.data.synth --limit 20 --out data/synthetic/pilot.jsonl
# 此处直接调库演示：
from src.data.synth import Synthesizer
syn = Synthesizer(limit=20, out="../data/synthetic/pilot.jsonl")
try:
    syn.run()
except Exception as e:
    print("合成失败：", e)
    print("检查 .env 的 SYNTH_API_KEY / SYNTH_BASE_URL / SYNTH_MODEL")

## 2. 抽检样本

In [ ]:
import json, random
from pathlib import Path
f = Path("../data/synthetic/pilot.jsonl")
if f.exists():
    rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
    print(f"共 {len(rows)} 条\n")
    for r in random.sample(rows, min(3, len(rows))):
        print("=" * 60)
        print("意图:", r.get("intent"), "| 情绪:", r.get("emotion"))
        print("问:", r.get("conversations", [{}])[0].get("value", "")[:80])
        print("答:", r.get("conversations", [{}])[-1].get("value", "")[:120])
else:
    print("先跑上面的试跑格子。")

## 3. 分布对照（合成 vs Day 8 计划）

In [ ]:
from collections import Counter
from src.data.taxonomy import TARGET_DISTRIBUTION
if f.exists():
    c = Counter(r.get("intent", "?") for r in rows)
    total = sum(c.values())
    print(f"{'意图':<12}{'实际':>6}{'占比':>8}")
    for intent, cnt in c.most_common():
        print(f"{intent:<12}{cnt:>6}{cnt/total:>7.1%}")
    print("\n→ 偏差最大的意图记进打卡，Day 10/11 补偿。")
else:
    print("试跑后再看。")

## 4. 抽检合格率打分表（人工，5 分钟）

对 20 条逐条打分：合格 / 语气雷同 / 幻觉参数 / 答非所问。
合格率 < 70% → 改 `SYSTEM_PROMPT` / personas → 重跑。**不要带病放量。**

## 5. 验收
- [ ] 20 条试跑 ≥70% 合格
- [ ] 2k 条正式跑完（终端命令），断点文件在
- [ ] 记账：花了多少钱 / 多少条 / 单条成本